In [1]:
from flask import Flask, render_template, request

In [2]:
import numpy as np
import pandas as pd

In [3]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
app = Flask(__name__)

In [8]:
# Absolute path example
df = pd.read_csv("recipe_final.csv")

df = df.dropna()



In [9]:
# Preprocess Ingredients
vectorizer = TfidfVectorizer()

vectorizer.fit(df["cuisine"] + " " + df["diet"] + " " + df["prep_time"] + " " + df["ingredients"])


KeyError: 'cuisine'

In [10]:
# Now transform each column using the same vocabulary
x_vect_cuisine = vectorizer.transform(df["cuisine"])
x_vect_diet = vectorizer.transform(df["diet"])
x_vect_time = vectorizer.transform(df["prep_time"])
X_ingredients = vectorizer.transform(df["ingredients"])


KeyError: 'cuisine'

In [11]:
# Combine transformed features
X_combined = np.hstack([x_vect_cuisine.toarray(), x_vect_diet.toarray(), 
                         x_vect_time.toarray(), X_ingredients.toarray()])


NameError: name 'x_vect_cuisine' is not defined

In [12]:
# Train KNN Model
knn = NearestNeighbors(n_neighbors=3, metric='euclidean')
knn.fit(X_combined)


NameError: name 'X_combined' is not defined

In [13]:
def recommend_recipes(input_features):
    input_ingredients_transformed_1 = vectorizer.transform([input_features[0]])
    input_ingredients_transformed_2 = vectorizer.transform([input_features[1]])
    input_ingredients_transformed_3 = vectorizer.transform([input_features[2]])
    input_ingredients_transformed_4 = vectorizer.transform([input_features[3]])
    input_combined = np.hstack([input_ingredients_transformed_1.toarray(), input_ingredients_transformed_2.toarray(),input_ingredients_transformed_3.toarray(),input_ingredients_transformed_4.toarray()])
    distances, indices = knn.kneighbors(input_combined)
    recommendations = df.iloc[indices[0]]
    return recommendations[['name', 'ingredients', 'image_url']].head(5
                                                                      )

In [14]:
# Function to truncate product name
def truncate(text, length):
    if len(text) > length:
        return text[:length] + "..."
    else:
        return text

In [15]:
@app.route('/', methods=['GET', 'POST'])
def index():
    if request.method == 'POST':
        cuisine = (request.form['cuisine'])
        diet = (request.form['diet'])
        prep_time = (request.form['prep_time'])
        ingredients = (request.form['ingredients'])

In [16]:
# cholesterol = float(request.form['cholesterol'])

In [17]:
# sodium = float(request.form['sodium'])

In [18]:
# fiber = float(request.form['fiber'])

In [19]:
# ingredients = request.form['ingredients']

In [20]:
input_features = [cuisine,diet,prep_time, ingredients]
recommendations = recommend_recipes(input_features)
return render_template('index.html', recommendations=recommendations.to_dict(orient='records'),truncate = truncate)
return render_template('index.html', recommendations=[])


NameError: name 'cuisine' is not defined

In [21]:
if __name__ == '__main__':
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

C:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
